In [1]:
# ===============================================
# Code Block 1 — Imports, Paths, Files
# ===============================================
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

IN_DIR  = "../data/processed"
EVENTS_FILE    = os.path.join(IN_DIR, "events_cleaned.csv")
CASES_FILE     = os.path.join(IN_DIR, "case_features.csv")
RESOURCES_FILE = os.path.join(IN_DIR, "resource_features.csv")
OUT_FILE       = os.path.join(IN_DIR, "municipality_features.csv")

# Check files
for f in [EVENTS_FILE, CASES_FILE, RESOURCES_FILE]:
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing required file: {f}")
print("All input files present.")


All input files present.


In [2]:
# ===============================================
# Code Block 2 — Load files (with parsing)
# ===============================================
events = pd.read_csv(EVENTS_FILE, parse_dates=["timestamp"], low_memory=False)
case_features = pd.read_csv(CASES_FILE, parse_dates=["case_start", "case_end"], low_memory=False)
resource_features = pd.read_csv(RESOURCES_FILE, low_memory=False)

print("Loaded shapes:")
print("events:", events.shape)
print("cases:", case_features.shape)
print("resources:", resource_features.shape)


Loaded shapes:
events: (262628, 14)
cases: (3183, 19)
resources: (64, 12)


In [3]:
# ===============================================
# Code Block 3 — Ensure municipality exists in case_features
# ===============================================
if "municipality" not in case_features.columns:
    print("municipality missing in case_features — deriving from events")
    mun = events.groupby("case_id")["municipality"].first().reset_index()
    case_features = case_features.merge(mun, on="case_id", how="left")

# Basic check
print("Municipality value counts (top):")
display(case_features["municipality"].value_counts().head(10))


Municipality value counts (top):


municipality
BPIC15_1    810
BPIC15_3    804
BPIC15_5    637
BPIC15_4    622
BPIC15_2    310
Name: count, dtype: int64

In [4]:
# ===============================================
# Code Block 4 — Case-level municipality aggregates
# ===============================================
# Ensure required columns exist or create placeholders
for c in ["cycle_time_hours", "wait_time_hours", "processing_time_hours", "rework_count", "total_cost", "variant_length"]:
    if c not in case_features.columns:
        case_features[c] = np.nan

# overdue handling: create if missing (default 0)
if "is_overdue" not in case_features.columns:
    case_features["is_overdue"] = 0

muni_case_agg = case_features.groupby("municipality").agg(
    n_cases                 = ("case_id", "nunique"),
    avg_cycle_hours         = ("cycle_time_hours", "mean"),
    avg_wait_hours          = ("wait_time_hours", "mean"),
    avg_processing_hours    = ("processing_time_hours", "mean"),
    avg_rework              = ("rework_count", "mean"),
    avg_cost                = ("total_cost", "mean"),
    std_cycle_hours         = ("cycle_time_hours", "std"),
    variant_complexity      = ("variant_length", "mean"),
    n_unique_variants       = ("variant", "nunique"),
    overdue_rate            = ("is_overdue", "mean")
).reset_index()

print("Case-level municipality aggregation complete. Sample:")
display(muni_case_agg.head())


Case-level municipality aggregation complete. Sample:


,municipality,n_cases,avg_cycle_hours,avg_wait_hours,avg_processing_hours,avg_rework,avg_cost,std_cycle_hours,variant_complexity,n_unique_variants,overdue_rate
0,BPIC15_1,810,2346.385508,2346.385508,0.0,3.541975,104027.618765,2863.190953,44.241975,796,0.0
1,BPIC15_2,310,3770.189789,3770.189789,0.0,4.164516,97524.171404,3248.651639,55.122581,309,0.0
2,BPIC15_3,804,1724.267544,1724.267544,0.0,3.767413,151821.381975,2524.220728,43.584577,769,0.0
3,BPIC15_4,622,2932.718776,2932.718776,0.0,185.311897,310319.419781,3373.633607,228.004823,618,0.0
4,BPIC15_5,637,2552.882832,2552.882832,0.0,4.029827,144841.536092,2853.616028,51.558870,635,0.0


In [5]:
# ===============================================
# Code Block 5 — Resource-level municipality aggregates (safe mapping)
# ===============================================
# resource_features may already have municipality column; ensure present
if "municipality" not in resource_features.columns:
    # fallback: map via events mode (safe)
    res_to_muni = (
        events[events["municipality"].notna()]
        .groupby("resource")["municipality"]
        .agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else np.nan)
        .rename("municipality")
    )
    resource_features = resource_features.merge(res_to_muni.reset_index(), on="resource", how="left")

resource_features["municipality"] = resource_features["municipality"].fillna("UNKNOWN_MUNI")

muni_res_agg = resource_features.groupby("municipality").agg(
    n_resources               = ("resource", "nunique"),
    avg_handover_in          = ("handover_in", "mean"),
    avg_handover_out         = ("handover_out", "mean"),
    avg_processing_hours_res = ("avg_inter_event_hours", "mean"),
    avg_wait_hours_res       = ("avg_wait_before_event_hours", "mean")
).reset_index()

print("Resource-level municipality aggregation complete. Sample:")
display(muni_res_agg.head())


Resource-level municipality aggregation complete. Sample:


,municipality,n_resources,avg_handover_in,avg_handover_out,avg_processing_hours_res,avg_wait_hours_res
0,BPIC15_1,21,120.333333,120.285714,106.129609,45.338299
1,BPIC15_2,6,496.333333,496.333333,65.873137,19.182486
2,BPIC15_3,20,186.150000,186.150000,64.611105,33.944504
3,BPIC15_4,9,342.666667,342.777778,81.202519,31.182385
4,BPIC15_5,8,438.000000,438.000000,54.468851,35.875489


In [6]:
# ===============================================
# Code Block 6 — Merge case + resource aggregates
# ===============================================
muni = muni_case_agg.merge(muni_res_agg, on="municipality", how="left")
print("Merged municipality table. Sample:")
display(muni.head())


Merged municipality table. Sample:


,municipality,n_cases,avg_cycle_hours,avg_wait_hours,avg_processing_hours,avg_rework,avg_cost,std_cycle_hours,variant_complexity,n_unique_variants,overdue_rate,n_resources,avg_handover_in,avg_handover_out,avg_processing_hours_res,avg_wait_hours_res
0,BPIC15_1,810,2346.385508,2346.385508,0.0,3.541975,104027.618765,2863.190953,44.241975,796,0.0,21,120.333333,120.285714,106.129609,45.338299
1,BPIC15_2,310,3770.189789,3770.189789,0.0,4.164516,97524.171404,3248.651639,55.122581,309,0.0,6,496.333333,496.333333,65.873137,19.182486
2,BPIC15_3,804,1724.267544,1724.267544,0.0,3.767413,151821.381975,2524.220728,43.584577,769,0.0,20,186.150000,186.150000,64.611105,33.944504
3,BPIC15_4,622,2932.718776,2932.718776,0.0,185.311897,310319.419781,3373.633607,228.004823,618,0.0,9,342.666667,342.777778,81.202519,31.182385
4,BPIC15_5,637,2552.882832,2552.882832,0.0,4.029827,144841.536092,2853.616028,51.558870,635,0.0,8,438.000000,438.000000,54.468851,35.875489


In [7]:
# ===============================================
# Code Block 7 — Handover per case per municipality (vectorized)
# ===============================================
# We reuse the events helper columns (actor_changed and prev_resource) — compute here if missing
if "prev_resource" not in events.columns or "actor_changed" not in events.columns:
    events = events.sort_values(["case_id", "timestamp"]).reset_index(drop=True)
    events["prev_resource"] = events.groupby("case_id")["resource"].shift(1)
    events["actor_changed"] = (events["resource"] != events["prev_resource"]) & events["prev_resource"].notna()

# Compute handovers per municipality: for events where actor_changed==True, group by municipality & case_id
handover_counts = (
    events[events["actor_changed"]]
    .groupby(["municipality", "case_id"])
    .size()
    .reset_index(name="handovers_in_case")
)

# Aggregate to municipality-level: average handovers per case
handover_muni = (
    handover_counts.groupby("municipality")["handovers_in_case"]
    .mean()
    .reset_index(name="handovers_per_case")
)

muni = muni.merge(handover_muni, on="municipality", how="left")
print("Added handovers_per_case. Sample:")
display(muni[["municipality", "handovers_per_case"]].head())


Added handovers_per_case. Sample:


,municipality,handovers_per_case
0,BPIC15_1,2527.0
1,BPIC15_2,2978.0
2,BPIC15_3,1861.5
3,BPIC15_4,3084.0
4,BPIC15_5,3504.0


In [8]:
# ===============================================
# Code Block 8 — Prepare metrics for normalization (handle NaNs and constants)
# ===============================================
neg_metrics = [
    "avg_cycle_hours",
    "avg_wait_hours",
    "avg_cost",
    "avg_rework",
    "overdue_rate",
    "avg_handover_out",
    "handovers_per_case"
]

# Ensure all columns exist
for col in neg_metrics:
    if col not in muni.columns:
        muni[col] = 0.0

# Replace infs and NaNs with zeros (or small sensible default)
muni[neg_metrics] = muni[neg_metrics].replace([np.inf, -np.inf], np.nan).fillna(0.0)

# If all values for a metric are identical, MinMaxScaler would return zeros; that's acceptable.
scaler = MinMaxScaler()

norm_cols = []
for col in neg_metrics:
    col_norm = col + "_norm"
    norm_cols.append(col_norm)
    # reshape to 2D array for scaler
    values = muni[[col]].values.astype(float)
    # safe-scaler: if constant, scaler will produce zeros
    try:
        muni[col_norm] = scaler.fit_transform(values).ravel()
    except Exception:
        # fallback: set zeros
        muni[col_norm] = 0.0

print("Normalization complete. Sample normalized columns:")
display(muni[[c for c in muni.columns if c.endswith("_norm")]].head())


Normalization complete. Sample normalized columns:


,avg_cycle_hours_norm,avg_wait_hours_norm,avg_cost_norm,avg_rework_norm,overdue_rate_norm,avg_handover_out_norm,handovers_per_case_norm
0,0.304077,0.304077,0.030562,0.000000,0.0,0.000000,0.405175
1,1.000000,1.000000,0.000000,0.003425,0.0,1.000000,0.679756
2,0.000000,0.000000,0.255162,0.001240,0.0,0.175149,0.000000
3,0.590663,0.590663,1.000000,1.000000,0.0,0.591659,0.744292
4,0.405008,0.405008,0.222361,0.002684,0.0,0.844878,1.000000


In [9]:
# ===============================================
# Code Block 9 — Compute performance index and rank
# ===============================================
weights = {
    "avg_cycle_hours_norm": 0.35,
    "avg_wait_hours_norm":  0.20,
    "avg_cost_norm":        0.15,
    "avg_rework_norm":      0.10,
    "overdue_rate_norm":    0.10,
    "avg_handover_out_norm": 0.05,
    "handovers_per_case_norm": 0.05
}

# Ensure all weighted cols exist (create zero if missing)
for k in weights.keys():
    if k not in muni.columns:
        muni[k] = 0.0

muni["performance_index_raw"] = 0.0
for col, w in weights.items():
    muni["performance_index_raw"] += muni[col] * w

# invert since lower is better
muni["performance_index"] = 1.0 - muni["performance_index_raw"]

# ranking (higher performance_index => rank 1)
muni["rank"] = muni["performance_index"].rank(ascending=False, method="min").astype(int)

print("Performance index & ranks computed. Top municipalities:")
display(muni[["municipality", "performance_index", "rank"]].sort_values("rank").head(10))


Performance index & ranks computed. Top municipalities:


,municipality,performance_index,rank
2,BPIC15_3,0.952844,1
0,BPIC15_1,0.807915,2
4,BPIC15_5,0.651379,3
1,BPIC15_2,0.365670,4
3,BPIC15_4,0.358338,5


In [10]:
# ===============================================
# Code Block 10 — Save municipality_features.csv
# ===============================================
muni.to_csv(OUT_FILE, index=False)
print("Saved municipality features to:", OUT_FILE)
print("Final shape:", muni.shape)


Saved municipality features to: ../data/processed\municipality_features.csv
Final shape: (5, 27)


In [ ]:
# Replace case_df with your case-level DataFrame name
# Ensure each column below exists in that DataFrame

case_metric_cols = [
    "case_cycle_time",          # case cycle time
    "active_processing_time",   # active processing time
    "waiting_time",             # waiting / idle time
    "structural_complexity",    # structural complexity per case
    "total_case_cost",          # total cost per case
    "event_count",              # number of events per case
    "case_hybrid_score",        # hybrid performance score per case
    "case_outlier_score",       # or case_efficiency_index, etc.
]

case_metric_titles = [
    "Case Cycle Time",
    "Active Processing Time",
    "Waiting Time",
    "Structural Complexity",
    "Total Cost per Case",
    "Event Count per Case",
    "Case Hybrid Performance Score",
    "Case Outlier / Efficiency Score",
]

fig_case, axes_case = plot_metric_dashboard(
    df=case_df,
    metric_cols=case_metric_cols,
    titles=case_metric_titles,
    n_cols=4,  # 2x4 grid → 8 tiles
    suptitle="Case Performance Dashboard",
    x_axis_label="Case index (sorted by metric)",
)